# Fairness theo vùng của hệ thống cảnh báo đơn trễ

Hệ thống của bài Intermediate 01 chọn top 5% đơn có nguy cơ trễ lúc giao vận chuyển để CSKH nhắn tin.
Đây là can thiệp **có lợi** cho khách: câu hỏi fairness là "khách ở vùng nào có đơn trễ mà không được
giúp?" (equal opportunity), và "vùng nào bị làm phiền nhầm nhiều?" (FPR).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật, train model, gọi hàm. Test bỏ qua chúng.
  Phần dựng dữ liệu và model đã viết sẵn (lặp lại bài Intermediate 01 ở dạng gọn).
- Chấm: `uv run pytest tracks/data-science/modules/15-ml-system-design/exercises/intermediate/03-fairness-slices -q`

In [ ]:
import numpy as np
import pandas as pd

## 1. Nhóm và chọn top k

In [ ]:
NORTH_NORTHEAST = {"AC", "AM", "AP", "PA", "RO", "RR", "TO",             # Bắc
                   "AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"}   # Đông Bắc


def add_region(df: pd.DataFrame, state_col: str = "customer_state") -> pd.DataFrame:
    """Bản sao của df thêm cột region: "N/NE" nếu bang thuộc NORTH_NORTHEAST, "SP" nếu là SP, còn lại "khác"."""
    out = df.copy()
    state = out[state_col]
    out["region"] = np.where(state.isin(NORTH_NORTHEAST), "N/NE", np.where(state == "SP", "SP", "khác"))
    return out


In [ ]:
def select_top_k(scores, k: int) -> np.ndarray:
    """Mảng bool cùng độ dài scores, đúng min(k, n) phần tử True ở các điểm cao nhất."""
    scores = np.asarray(scores, dtype=float)
    mask = np.zeros(len(scores), dtype=bool)
    mask[np.argsort(-scores, kind="stable")[:max(0, min(k, len(scores)))]] = True
    return mask


## 2. Báo cáo theo nhóm

In [ ]:
def slice_report(df: pd.DataFrame, group_col: str, label_col: str, selected_col: str) -> pd.DataFrame:
    """Mỗi nhóm một dòng (index sắp tăng). Cột: n, prevalence, selection_rate, precision, recall, fpr,
    share_of_selected (phần của nhóm trong tổng số được chọn), share_of_positives (phần trong tổng ca dương).
    Mẫu số bằng 0 thì nan.
    """
    y = df[label_col].astype(bool)
    s = df[selected_col].astype(bool)
    total_selected, total_pos = s.sum(), y.sum()
    rows = {}
    for g, idx in df.groupby(group_col, sort=True).groups.items():
        yg, sg = y.loc[idx], s.loc[idx]
        ratio = lambda a, b: float(a / b) if b else float("nan")
        rows[g] = {"n": len(idx), "prevalence": float(yg.mean()), "selection_rate": float(sg.mean()),
                   "precision": ratio((yg & sg).sum(), sg.sum()), "recall": ratio((yg & sg).sum(), yg.sum()),
                   "fpr": ratio((~yg & sg).sum(), (~yg).sum()),
                   "share_of_selected": ratio(sg.sum(), total_selected),
                   "share_of_positives": ratio(yg.sum(), total_pos)}
    out = pd.DataFrame.from_dict(rows, orient="index")
    out.index.name = group_col
    return out


In [ ]:
def disparity(report: pd.DataFrame, metric: str, higher_is_better: bool = True) -> dict:
    """Chênh lệch giữa các nhóm của một cột trong slice_report (bỏ qua nan):
    ratio = min / max, difference = max − min, worst_group = nhóm có giá trị nhỏ nhất nếu higher_is_better
    (recall, precision), lớn nhất nếu không (fpr).
    """
    v = report[metric].dropna()
    return {"ratio": float(v.min() / v.max()), "difference": float(v.max() - v.min()),
            "worst_group": v.idxmin() if higher_is_better else v.idxmax()}


## 3. Một cách giảm chênh lệch

Chia ngân sách theo **nhu cầu** của từng vùng (số đơn trễ kỳ vọng = tổng xác suất), thay vì một bảng
xếp hạng chung.

In [ ]:
def allocate_by_need(scores, groups, k: int) -> np.ndarray:
    """Chia ngân sách k cho các nhóm theo tỉ lệ số ca dương kỳ vọng (tổng scores của nhóm), làm tròn bằng
    largest remainder (phần nguyên trước, phần dư lớn nhất được thêm 1), rồi chọn top trong từng nhóm.
    Trả mảng bool, đúng k phần tử True (giả định k <= n và mỗi nhóm đủ phần tử).
    """
    scores = np.asarray(scores, dtype=float)
    groups = np.asarray(groups)
    labels = np.unique(groups)
    need = np.array([scores[groups == g].sum() for g in labels])
    exact = need / need.sum() * k
    quota = np.floor(exact).astype(int)
    for i in np.argsort(-(exact - quota), kind="stable")[: k - quota.sum()]:
        quota[i] += 1
    mask = np.zeros(len(scores), dtype=bool)
    for g, q in zip(labels, quota):
        idx = np.flatnonzero(groups == g)
        mask[idx[np.argsort(-scores[idx], kind="stable")[:q]]] = True
    return mask


In [ ]:
# explore: dữ liệu Olist thật (tải bằng: uv run python data/download_olist.py) và model lúc giao vận chuyển
from pathlib import Path
import lightgbm as lgb

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data/raw"
dates = ["order_purchase_timestamp", "order_delivered_carrier_date",
         "order_delivered_customer_date", "order_estimated_delivery_date"]
o = pd.read_csv(RAW / "olist_orders_dataset.csv", parse_dates=dates)
o = o[(o.order_status == "delivered") & o.order_delivered_customer_date.notna()]
o = o.merge(pd.read_csv(RAW / "olist_customers_dataset.csv")[["customer_id", "customer_state"]], on="customer_id")
it = pd.read_csv(RAW / "olist_order_items_dataset.csv").merge(
    pd.read_csv(RAW / "olist_sellers_dataset.csv")[["seller_id", "seller_state"]], on="seller_id")
it = it.merge(o[["order_id", "customer_state"]], on="order_id")
it["same"] = it.seller_state == it.customer_state
agg = it.groupby("order_id").agg(n_items=("order_item_id", "size"), n_sellers=("seller_id", "nunique"),
                                 freight=("freight_value", "sum"), same_state=("same", "all"))
o = o.merge(agg, on="order_id", how="left")
day = pd.Timedelta(days=1)
df = pd.DataFrame({
    "purchase_ts": o.order_purchase_timestamp,
    "late": (o.order_delivered_customer_date > o.order_estimated_delivery_date).astype(int),
    "promised_days": (o.order_estimated_delivery_date - o.order_purchase_timestamp) / day,
    "same_state": o.same_state.fillna(False).astype(int), "log_freight": np.log1p(o.freight.fillna(0)),
    "n_items": o.n_items.fillna(0), "n_sellers": o.n_sellers.fillna(0),
    "customer_state": o.customer_state,
    "handling_days": (o.order_delivered_carrier_date - o.order_purchase_timestamp) / day,
    "remaining_days": (o.order_estimated_delivery_date - o.order_delivered_carrier_date) / day,
}).reset_index(drop=True)
train = df[df.purchase_ts < "2018-01-01"]
test = df[(df.purchase_ts >= "2018-04-01") & (df.purchase_ts < "2018-09-01")].reset_index(drop=True)
states = pd.CategoricalDtype(sorted(train.customer_state.unique()))
base = ["promised_days", "same_state", "log_freight", "n_items", "n_sellers", "customer_state"]

def score(feats):
    X = lambda d: d[feats].assign(customer_state=d.customer_state.astype(states))
    m = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, min_child_samples=50, verbose=-1, random_state=0)
    return m.fit(X(train), train.late).predict_proba(X(test))[:, 1]

test = add_region(test)
test["p_carrier"] = score(base + ["handling_days", "remaining_days"])
test["p_purchase"] = score(base)
K = int(0.05 * len(test))
print(len(test), K)

In [ ]:
# explore: tỉ lệ trễ theo vùng trên toàn bộ đơn giao xong
add_region(df).groupby("region").late.agg(["mean", "size"])

In [ ]:
# explore: top 5% theo model lúc giao vận chuyển
test["sel_carrier"] = select_top_k(test.p_carrier, K)
rep = slice_report(test, "region", "late", "sel_carrier")
display(rep.round(3))
for metric in ["precision", "recall"]:
    print(metric, disparity(rep, metric))
print("fpr", disparity(rep, "fpr", higher_is_better=False))

In [ ]:
# explore: model lúc đặt hàng (yếu hơn) dựa vào vùng nhiều hơn
test["sel_purchase"] = select_top_k(test.p_purchase, K)
slice_report(test, "region", "late", "sel_purchase").round(3)

In [ ]:
# explore: chia ngân sách theo nhu cầu, so recall và tổng giá trị (tin nhắn 0.5 R$, V = 30 R$, Δ = 0.15)
test["sel_need"] = allocate_by_need(test.p_carrier, test.region, K)
display(slice_report(test, "region", "late", "sel_need").round(3))
for col in ["sel_carrier", "sel_need"]:
    tp = test.late[test[col]].sum()
    print(col, tp, tp * 0.15 * 30 - K * 0.5)

## 4. Câu hỏi

1. Với top 5% chung, vùng nào có recall thấp nhất? Vùng nào bị cảnh báo nhầm nhiều nhất?
2. Vì sao Bắc / Đông Bắc có selection rate cao nhất nhưng precision thấp nhất? Đây là lỗi của model hay
   hệ quả của tỉ lệ nền?
3. So model lúc đặt hàng và lúc giao vận chuyển: model nào "dựa vào vùng" nhiều hơn? Dấu hiệu là gì?
4. Chia theo nhu cầu đổi recall các vùng ra sao, mất bao nhiêu giá trị? Bạn có khuyến nghị nó không?
5. Bỏ cột `customer_state` khỏi model có làm hệ thống công bằng hơn không? Vì sao?

## Trả lời (lời giải)

1. Bang khác: recall 17.9% (São Paulo 43.6%, Bắc / Đông Bắc 31.7%). Bắc / Đông Bắc có FPR 15.2%, gấp
   khoảng 8 lần hai vùng kia.
2. Tỉ lệ trễ nền ở Bắc / Đông Bắc cao (11.1% trên test, 13.6% trên toàn bộ), nên model đẩy điểm cả vùng
   lên; trong vùng đó tín hiệu phân biệt từng đơn yếu hơn, nên nhiều cảnh báo nhầm. Khi tỉ lệ nền khác
   nhau, không thể vừa calibrate vừa cân bằng FPR/TPR (Kleinberg; Chouldechova): đây là hệ quả toán học,
   không chỉ là lỗi model.
3. Model lúc đặt hàng: Bắc / Đông Bắc nhận 50% cảnh báo dù chỉ chiếm 18.9% số đơn trễ, recall São Paulo
   chỉ 4.4%. Model yếu thay tín hiệu từng đơn bằng proxy nhóm.
4. Chia theo nhu cầu: recall bang khác 17.9% → 23.5%, São Paulo gần như giữ (44.2%), Bắc / Đông Bắc
   31.7% → 20.8%; tổng giá trị 2 022.5 → 2 036 R$ (tăng nhẹ). Có thể khuyến nghị, nhưng phải nói rõ
   Bắc / Đông Bắc mất recall: lựa chọn này là quyết định của business, DS đặt nó lên bàn.
5. Không chắc. Phí ship, số ngày hẹn và cùng bang mã hoá phần lớn thông tin địa lý. Bỏ cột nhóm thường
   chỉ làm khó đo lường; cách tốt hơn là giữ cột nhóm để **đo**, rồi quyết định chính sách theo nhóm.